#### Standard RAG uses Vector Search, which works like searching a library by matching keywords or general meaning. It’s good for finding specific facts, but not for making connections. That’s where GraphRAG comes in. Instead of seeing your data as separate documents, GraphRAG views it as a network of connected facts.

#### GraphRAG uses a Knowledge Graph, which is a network of entities (nodes) and relationships (edges). This lets it move from one fact to another and find hidden connections that vector search can’t catch.

Pipeline:

1. Read text.
2. Extract relationships (Subject -> Predicate -> Object).
3. Build a Graph using NetworkX.
4. Retrieve context by walking the graph (Multi-hop reasoning).
5. Answer a question based on that deep context.

In [1]:
import networkx as nx
import json
import os
from huggingface_hub import InferenceClient

/root/opt/ra-i-g/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading Remote LLM (instead of local Ollama, which is not working on the server or small GPUs)
client = InferenceClient(token=os.getenv("HF_TOKEN"))
MODEL = "meta-llama/Meta-Llama-3-8B-Instruct" # For production, use GPT-4o or Claude 3.5 Sonnet for better accuracy

def ask_llm(prompt, max_tokens=500):
    """Helper to call the LLM"""
    result = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        max_tokens=max_tokens,
        temperature=0.0, # facts, no creativity
    )
    return result.choices[0].message.content.strip()


In [3]:
# Turnin text into data : extracting triples from text
# We can’t put raw text straight into a graph; we need triples. A triple is the basic unit of a knowledge graph: (Head) -> [Relation] -> (Tail).

def extract_triples(text):
    """Replaces the LangChain extraction_chain"""
    prompt = f"""You are an expert knowledge graph builder.
                Extract entities and relationships from the text.
                Return ONLY a JSON list. No explanation, no markdown, no backticks.
                Each item must contain:
                - "head": source entity
                - "relation": relationship
                - "tail": target entity

                Text:
                {text}

                Output JSON:"""
    
    response = ask_llm(prompt)
    # Clean up in case the model wraps in backticks
    response = response.strip().strip('`')
    if response.startswith('json'):
        response = response[4:]
    return json.loads(response)

In [4]:
# Testing on the example bout the Moon
text = """
The Moon orbits Earth.The Moon has an atmosphere called the Exosphere.Apollo 11 landed on the Moon.The Moon has a crater named the South Pole-Aitken Basin.Earth's Moon is classified as a natural satellite.
"""

print("\n Extracting knowledge graph triples...\n")
triples = extract_triples(text)
print(triples)


 Extracting knowledge graph triples...

[{'head': 'The Moon', 'relation': 'orbits', 'tail': 'Earth'}, {'head': 'The Moon', 'relation': 'has', 'tail': 'the Exosphere'}, {'head': 'Apollo 11', 'relation': 'landed', 'tail': 'the Moon'}, {'head': 'The Moon', 'relation': 'has', 'tail': 'the South Pole-Aitken Basin'}, {'head': 'Earth', 'relation': 'has', 'tail': 'the Moon'}]


In [5]:
# Building Knowledge Graph
kg = nx.DiGraph() # DiGraph means "Directed Graph" (arrows point one way)

def build_knowledge_graph(triples):
    for item in triples:
        head = item.get("head")
        tail = item.get("tail")
        relation = item.get("relation")

        if head and tail:
            kg.add_node(head)
            kg.add_node(tail)
            kg.add_edge(head, tail, label=relation)

build_knowledge_graph(triples)

print("\n Nodes in Graph:")
print(list(kg.nodes()))


 Nodes in Graph:
['The Moon', 'Earth', 'the Exosphere', 'Apollo 11', 'the Moon', 'the South Pole-Aitken Basin']


In [6]:
# Multi-Hop Retrieval: Finding paths between entities
def retrieve_graph_context(entity, max_depth=2):
    context = set()
    visited_nodes = set()

    def dfs(node, depth):
        if depth > max_depth:
            return
        visited_nodes.add(node)

        # Checking Outgoing edges (What does this node do?)
        for neighbor in kg.successors(node):
            relation = kg.get_edge_data(node, neighbor)["label"]
            context.add(f"{node} {relation} {neighbor}")
            if neighbor not in visited_nodes:
                dfs(neighbor, depth + 1)

        # Checking Incoming edges (Who interacts with this node?)
        for predecessor in kg.predecessors(node):
            relation = kg.get_edge_data(predecessor, node)["label"]
            context.add(f"{predecessor} {relation} {node}")
            if predecessor not in visited_nodes:
                dfs(predecessor, depth + 1)

    if entity in kg.nodes:
        dfs(entity, 1) # Starts the traversal

    return ". ".join(context)

In [ ]:
# RAG
# Feeding that rich, interconnected context back to the LLM to answer the user’s question

def graph_rag_answer(entity, question, max_depth=3):
    """Replaces the LangChain rag_chain. Ask for a depth of at least 3 to catch distant connections"""
    graph_context = retrieve_graph_context(entity, max_depth=max_depth)
    
    print(f"\nRetrieved Graph Context:\n{graph_context}\n")
    
    prompt = f"""Answer the question using ONLY the context below.

                Context:
                {graph_context}

                Question:
                {question}

                Answer:"""
    
    return ask_llm(prompt, max_tokens=200)

In [ ]:
# Asking a Multi-Hop reasoning question

entity = "Apollo 11"
question = "On which natural satellite did Apollo land?"
answer = graph_rag_answer(entity, question)

print(f"\nFinal Answer:\n{answer}")


Retrieved Graph Context:
Earth has the Moon. Apollo 11 landed the Moon. The Moon orbits Earth


Final Answer:
The answer is: The Moon.
